# 1.10 · 数据仓库 / Data Warehouse

> **课程定位 / Where this fits**
> **Part 1 第 10 课**——DS 入职后每天查的表，90% 不是业务库原表，而是**数仓里的事实表和维度表**。这一课讲清楚它们为什么长这样：星型模型、粒度、代理键、缓慢变化维。
> **Part 1, lesson 10** — 90% of the tables a DS queries daily are warehouse fact/dim tables, not raw OLTP tables. This lesson explains why they're shaped that way.

> 💡 **面试相关 / Interview-relevant**
> - "星型 vs 雪花模型" ★★★★（数据岗必考）
> - "事实表的粒度（grain）是什么" ★★★★
> - "SCD Type 2 怎么实现" ★★★★（分析工程师必考）
> - "ETL vs ELT" ★★★
> - "为什么用代理键不用业务键" ★★★

---

## 学习目标 / Learning Objectives

1. 从存储布局解释 **OLTP（行存）vs OLAP（列存）** 为什么各快各的。
   Explain row-store vs column-store from the storage layout up.
2. 设计一个**星型模型**：选事实、定**粒度**、拆维度。
   Design a star schema: pick facts, declare the grain, carve dimensions.
3. 说出**代理键**对业务键的三个优势。
   Give three reasons surrogate keys beat natural keys.
4. **亲手实现 SCD Type 2**（拉链表），并查询"某历史时点的状态"。
   Implement SCD Type 2 by hand and run an as-of query.
5. 区分 **ETL vs ELT**，知道现代栈为什么倒向 ELT。
   Distinguish ETL vs ELT and why the modern stack went ELT.
6. 知道三大云数仓的差异点一句话版。
   One-liner differences among the big-three cloud warehouses.

---

## 目录 / Table of Contents

1. [OLTP vs OLAP：正式分家 / The Formal Split](#1)
2. [行存 vs 列存 / Row Store vs Column Store](#2)
3. [星型模型 ⭐ / The Star Schema](#3)
4. [粒度：事实表的第一决定 ⭐ / Grain](#4)
5. [动手建星型模型 / Hands-on: Build the Star](#5)
6. [代理键 / Surrogate Keys](#6)
7. [缓慢变化维 SCD ⭐ / Slowly Changing Dimensions](#7)
8. [雪花模型 & One Big Table / Snowflake Schema & OBT](#8)
9. [ETL vs ELT](#9)
10. [云数仓三巨头 / The Big-Three Cloud Warehouses](#10)
11. [小结 / Summary](#11)


<a id="1"></a>
## 1. OLTP vs OLAP：正式分家 / The Formal Split

| 维度 / Aspect | OLTP（交易库）| OLAP（分析库 / 数仓）|
|---|---|---|
| 典型操作 | "给订单 #42 改状态" | "过去 12 个月每地区月销售额" |
| 访问模式 | 按 key 取**少数行的全部列** | 扫**海量行的少数列** |
| 读写比 | 读写均衡，高并发小事务 | 重读，批量写入 |
| 数据建模 | **范式化**（避免更新异常）| **反范式化**（避免 JOIN 成本）|
| 存储 | 行存 + B-tree | **列存 + 压缩** |
| 例子 | Postgres, MySQL | Snowflake, BigQuery, DuckDB |

### 为什么不能一个库两用 / Why not one DB for both

分析查询一来就扫几亿行 → 把交易库的缓存、I/O、锁全打乱 → **线上下单变慢**。
所以工业标准是：**业务库照常跑，数据"搬"到数仓里做分析**（怎么搬 = ETL/ELT，第 9 节）。
One analytics scan trashes the OLTP cache and I/O, slowing real orders. So industry copies data out to a warehouse — how it's copied is ETL/ELT (section 9).


<a id="2"></a>
## 2. 行存 vs 列存 / Row Store vs Column Store

同一张表，磁盘上两种摆法：
Same table, two physical layouts:

```
逻辑表:  (id, name, country, amount)
         (1, Alice, US, 120)
         (2, Bob,   UK,  75)
         (3, Carol, US, 300)

行存 (OLTP):   [1|Alice|US|120] [2|Bob|UK|75] [3|Carol|US|300]
               → 取"订单 #2 的全部信息" = 读 1 块 ✅
               → 算"SUM(amount)" = 把每行全读出来再扔掉 3/4 ❌

列存 (OLAP):   [1|2|3] [Alice|Bob|Carol] [US|UK|US] [120|75|300]
               → 算"SUM(amount)" = 只读 amount 那一段 ✅
               → 取"订单 #2 的全部信息" = 跳 4 个地方拼一行 ❌
```

### 列存的第二重红利：压缩 / The second dividend: compression

同一列的数据类型相同、值域相近 → 压缩率极高：
Same-typed, similar-valued data compresses brutally well:

- `country` 列只有 5 个值 → **字典编码**（每行 1 字节都不用）
- 排序后的时间戳 → **delta 编码**（存差值）
- 重复值 → **RLE**（run-length，"US × 80000 次"存成一对）

**压缩 10-30× 常见** → 同样的 I/O 带宽读 10-30 倍的数据 → 这就是 DuckDB / Snowflake 扫几亿行还很快的原因（再叠加 1.7 节讲过的 zone maps 跳块）。
10-30x compression is routine — the same I/O bandwidth moves 10-30x more data. Stack zone maps (lesson 1.7) on top and that's why columnar engines fly.


<a id="3"></a>
## 3. 星型模型 ⭐ / The Star Schema

**Kimball 维度建模**——过去 30 年数仓的事实标准。
Kimball dimensional modeling — the de-facto warehouse standard for 30 years.

```
                    ┌──────────────┐
                    │  dim_date    │
                    │  date_key PK │
                    │  year, month │
                    │  is_weekend  │
                    └──────┬───────┘
                           │
┌──────────────┐    ┌──────┴────────────┐    ┌────────────────┐
│ dim_customer │    │   fact_sales      │    │  dim_track     │
│ customer_key ├────┤ date_key     FK   ├────┤ track_key  PK  │
│ name         │    │ customer_key FK   │    │ title          │
│ country      │    │ track_key    FK   │    │ genre          │
│ segment      │    │ ───────────────   │    │ artist_name    │
└──────────────┘    │ quantity     ⊕    │    │ album_title    │
                    │ revenue      ⊕    │    └────────────────┘
                    └───────────────────┘
                      中心 = 事实表（数字）
                      周围 = 维度表（描述）→ 画出来像星星 ⭐
```

### 两类表的分工 / The division of labor

| | 事实表 / Fact | 维度表 / Dimension |
|---|---|---|
| 存什么 | **可加的数字**（quantity, revenue）+ 外键 | **描述性属性**（name, genre, country）|
| 行数 | 巨大（亿级）| 小（千~百万级）|
| 增长 | 每天追加 | 缓慢变化 |
| 查询角色 | `SUM / COUNT` 的对象 | `WHERE / GROUP BY` 的对象 |

### 为什么不规范化维度 / Why dimensions stay denormalized

`dim_track` 里 artist_name 和 album_title **直接展开**（而不是再建 dim_artist 外键）——**违反三范式，故意的**：
- 分析查询少 JOIN 一两层 = 快 + SQL 好写
- 维度表本来就小，冗余几 MB 无所谓
- 业务用户（看板拖拽）不用理解 5 层关系

Dimensions deliberately violate 3NF: fewer JOINs, simpler SQL, and the redundancy costs megabytes, not gigabytes.


<a id="4"></a>
## 4. 粒度：事实表的第一决定 ⭐ / Grain

**粒度 = 事实表的一行代表什么**。Kimball 的第一条戒律：**先声明粒度，再做任何事**。
Grain = what one fact row represents. Kimball's first commandment: declare the grain before anything else.

| 粒度声明 / Grain | 一行 = | 能回答 | 不能回答 |
|---|---|---|---|
| 每张发票一行 | 一笔订单 | 订单数、客单价 | "哪首歌卖得好"（歌曲信息被压扁了）|
| **每张发票的每行明细一行** ⭐ | 一次"买某首歌" | 上面全部 + 歌曲/类型分析 | 单次播放行为 |
| 每次播放一行 | 一次播放事件 | 全部 + 收听时长分析 | —（最细）|

### 两条铁律 / Two iron rules

1. **取业务允许的最细粒度**——粗粒度永远无法事后变细，细粒度随时可以 `GROUP BY` 变粗。
   Take the finest grain the business allows — you can always roll up, never drill down past the grain.
2. **一张事实表只有一个粒度**——混粒度（订单行 + 订单头挤一张表）= 重复计数灾难（1.2 节的行数爆炸又来了）。
   One fact table, one grain. Mixed grains = double-counting disasters.

> 💡 **面试题"事实表的 grain 是什么"** 的满分回答：先背定义，再举"声明粒度避免重复计数"的例子。


<a id="5"></a>
## 5. 动手建星型模型 / Hands-on: Build the Star

把 1.1–1.5 节的 Mini Music Store（**OLTP 范式化 5 表**）改造成**数仓星型模型**——这正是现实里分析工程师的日常。
We transform the normalized 5-table OLTP Music Store into a warehouse star — exactly what analytics engineers do daily.


In [ ]:
import duckdb
import pandas as pd

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

conn = duckdb.connect()

# ---- 先重建 OLTP 源表（来自 1.1 节）/ Rebuild the OLTP source ----
conn.sql("""
CREATE TABLE artist (artist_id INT, name VARCHAR, country VARCHAR);
INSERT INTO artist VALUES
    (1,'The Beatles','UK'),(2,'Pink Floyd','UK'),(3,'Miles Davis','US'),
    (4,'Daft Punk','FR'),(5,'Radiohead','UK'),(6,'Anonymous Artist',NULL);

CREATE TABLE album (album_id INT, title VARCHAR, artist_id INT, year INT);
INSERT INTO album VALUES
    (1,'Abbey Road',1,1969),(2,'The Dark Side of the Moon',2,1973),
    (3,'The Wall',2,1979),(4,'Kind of Blue',3,1959),(5,'Discovery',4,2001),
    (6,'Random Access Memories',4,2013),(7,'OK Computer',5,1997),
    (8,'Demos (unreleased)',6,2024);

CREATE TABLE track (track_id INT, name VARCHAR, album_id INT, genre VARCHAR,
                    seconds INT, price DECIMAL(4,2));
INSERT INTO track VALUES
    (1,'Come Together',1,'Rock',259,0.99),(2,'Something',1,'Rock',182,0.99),
    (3,'Here Comes the Sun',1,'Rock',185,0.99),(4,'Time',2,'Rock',413,1.29),
    (5,'Money',2,'Rock',382,1.29),(6,'Us and Them',2,'Rock',460,1.29),
    (7,'Another Brick in the Wall',3,'Rock',239,1.29),
    (8,'Comfortably Numb',3,'Rock',382,1.29),(9,'So What',4,'Jazz',545,1.49),
    (10,'Freddie Freeloader',4,'Jazz',586,1.49),(11,'Blue in Green',4,'Jazz',337,1.49),
    (12,'One More Time',5,'Electronic',320,1.29),(13,'Aerodynamic',5,'Electronic',213,1.29),
    (14,'Digital Love',5,'Electronic',301,1.29),(15,'Get Lucky',6,'Electronic',369,1.29),
    (16,'Instant Crush',6,'Electronic',337,1.29),(17,'Lose Yourself to Dance',6,'Electronic',353,1.29),
    (18,'Paranoid Android',7,'Rock',384,1.29),(19,'Karma Police',7,'Rock',261,1.29),
    (20,'No Surprises',7,'Rock',228,1.29),(21,'Untitled Demo 1',8,'Rock',180,0.50),
    (22,'Untitled Demo 2',8,'Rock',195,0.50);

CREATE TABLE customer (customer_id INT, name VARCHAR, country VARCHAR, email VARCHAR);
INSERT INTO customer VALUES
    (1,'Alice Chen','US','alice@example.com'),(2,'Bob Smith','UK','BOB@example.com'),
    (3,'Charlie Davis','US','charlie@example.com'),(4,'Diana Park','DE','diana@example.com'),
    (5,'Ethan Miller','US','ethan@example.com'),(6,'Fiona Wong','JP',NULL);

CREATE TABLE invoice (invoice_id INT, customer_id INT, track_id INT,
                      invoice_date DATE, quantity INT);
INSERT INTO invoice VALUES
    (1,1,1,DATE '2026-01-05',1),(2,1,9,DATE '2026-01-05',2),
    (3,2,4,DATE '2026-01-10',1),(4,2,18,DATE '2026-01-10',1),
    (5,3,5,DATE '2026-02-12',1),(6,3,6,DATE '2026-02-12',1),
    (7,3,7,DATE '2026-02-12',3),(8,4,15,DATE '2026-02-20',1),
    (9,4,12,DATE '2026-02-20',1),(10,4,13,DATE '2026-02-20',1),
    (11,5,9,DATE '2026-03-01',1),(12,5,10,DATE '2026-03-01',1),
    (13,5,11,DATE '2026-03-01',1),(14,5,4,DATE '2026-03-05',2),
    (15,1,18,DATE '2026-03-15',1),(16,1,19,DATE '2026-03-15',1),
    (17,2,15,DATE '2026-04-01',1),(18,3,14,DATE '2026-04-10',2),
    (19,4,8,DATE '2026-05-02',1),(20,5,1,DATE '2026-05-20',1);
""")
print("OLTP source ready:", conn.sql("SHOW TABLES").df()["name"].tolist())


In [ ]:
# ---- Step 1: dim_track —— 把 track/album/artist 三表"拍平"成一个维度 ----
# Flatten track + album + artist into ONE denormalized dimension
conn.sql("""
CREATE TABLE dim_track AS
SELECT
    ROW_NUMBER() OVER (ORDER BY t.track_id)   AS track_key,    -- 代理键 / surrogate key
    t.track_id                                 AS track_nk,     -- 业务键留着 / natural key kept
    t.name                                     AS title,
    t.genre,
    t.seconds,
    a.title                                    AS album_title,
    a.year                                     AS album_year,
    ar.name                                    AS artist_name,
    COALESCE(ar.country, 'Unknown')            AS artist_country
FROM track t
JOIN album a   ON t.album_id  = a.album_id
JOIN artist ar ON a.artist_id = ar.artist_id;
""")
print(conn.sql("SELECT * FROM dim_track LIMIT 4").df())


In [ ]:
# ---- Step 2: dim_date —— 日期维度（数仓标配）----
# The date dimension: pre-computed calendar attributes
conn.sql("""
CREATE TABLE dim_date AS
SELECT
    CAST(STRFTIME(d, '%Y%m%d') AS INT)  AS date_key,         -- 20260105 形式的"智能键"
    d                                    AS full_date,
    EXTRACT(YEAR FROM d)                 AS year,
    EXTRACT(MONTH FROM d)                AS month,
    STRFTIME(d, '%B')                    AS month_name,
    EXTRACT(QUARTER FROM d)              AS quarter,
    ISODOW(d)                            AS day_of_week,      -- 1=Mon
    ISODOW(d) IN (6, 7)                  AS is_weekend
FROM (
    SELECT UNNEST(GENERATE_SERIES(DATE '2026-01-01', DATE '2026-12-31', INTERVAL 1 DAY)) AS d
);
""")
print(conn.sql("SELECT * FROM dim_date WHERE month = 1 LIMIT 4").df())
print("\n为什么要日期维度？让 '周末 vs 工作日销售' 这种查询不用每次现算日期函数")


In [ ]:
# ---- Step 3: dim_customer ----
conn.sql("""
CREATE TABLE dim_customer AS
SELECT
    ROW_NUMBER() OVER (ORDER BY customer_id) AS customer_key,
    customer_id                              AS customer_nk,
    name,
    country,
    CASE WHEN email IS NULL THEN 'no_email' ELSE 'has_email' END AS contactable
FROM customer;
""")

# ---- Step 4: fact_sales —— 粒度声明：每张发票的每行明细一行 ----
# GRAIN: one row per invoice line item
conn.sql("""
CREATE TABLE fact_sales AS
SELECT
    CAST(STRFTIME(i.invoice_date, '%Y%m%d') AS INT) AS date_key,
    dc.customer_key,
    dt.track_key,
    i.invoice_id                                     AS invoice_nk,   -- 退化维度 / degenerate dim
    i.quantity,
    i.quantity * t.price                             AS revenue
FROM invoice i
JOIN track        t  ON i.track_id    = t.track_id
JOIN dim_customer dc ON i.customer_id = dc.customer_nk
JOIN dim_track    dt ON i.track_id    = dt.track_nk;
""")
print(conn.sql("SELECT * FROM fact_sales LIMIT 5").df())
print(f"\nfact rows: {conn.sql('SELECT COUNT(*) FROM fact_sales').fetchone()[0]} (= 20 张发票行，粒度正确)")


In [ ]:
# ---- 星型模型的回报：分析查询变得又快又好写 ----
# The payoff: analytics queries get simple AND fast
# "每季度、每 genre、周末 vs 工作日的销售额" —— 三个维度任意切
conn.sql("""
    SELECT
        dd.quarter,
        dt.genre,
        dd.is_weekend,
        SUM(f.revenue)  AS revenue,
        SUM(f.quantity) AS units
    FROM fact_sales f
    JOIN dim_date  dd USING (date_key)
    JOIN dim_track dt USING (track_key)
    GROUP BY ALL
    ORDER BY quarter, revenue DESC;
""").df()


**注意这条查询的形状**：`FROM fact JOIN dim JOIN dim ... GROUP BY 维度属性, SUM(事实)` ——
**数仓里 95% 的查询都长这样**。BI 工具（Tableau / Looker）的拖拽底层生成的就是这种 SQL。
Note the query shape: fact JOIN dims, GROUP BY dim attributes, SUM facts. 95% of warehouse queries look exactly like this — it's what BI tools generate under the hood.


<a id="6"></a>
## 6. 代理键 / Surrogate Keys

刚才建维度时我们都加了 `ROW_NUMBER()` 生成的 `_key` 列，而不是直接用业务的 `customer_id`。**为什么多此一举？**
We added ROW_NUMBER surrogate keys instead of reusing business IDs. Why bother?

| 理由 / Reason | 场景 / Scenario |
|---|---|
| **1. 业务键会变** | 公司换 CRM，customer_id 全部重编 → 代理键隔离了这种冲击 |
| **2. 多源合并** | 两个收购来的系统都有 customer_id=42 → 代理键避免撞车 |
| **3. SCD 需要** ⭐ | 同一个客户的"历史版本"需要**多行** → 业务键没法当主键了（下一节）|
| 4. 性能 | 整数 JOIN 比长字符串业务键（如 email）快 |

> 💡 **命名约定**：`*_key` = 代理键（数仓内部），`*_nk` 或 `*_id` = 业务键（natural key，从源系统带来）。两个都存：代理键用来 JOIN，业务键用来对账。
> Convention: `*_key` = surrogate (warehouse-internal), `*_nk` = natural key (from source). Keep both — JOIN on surrogate, reconcile on natural.


<a id="7"></a>
## 7. 缓慢变化维 SCD ⭐ / Slowly Changing Dimensions

**问题**：客户 Alice 2026 年 3 月从 US 搬到 CA。她 1 月的订单算 US 销售还是 CA 销售？
**The problem**: Alice moves US→CA in March. Do her January orders count as US or CA sales?

| 策略 / Type | 做法 / Action | 历史 / History | 1 月订单归属 |
|---|---|---|---|
| **Type 1** | 直接覆盖 country 字段 | ❌ 丢失 | CA（**错！**历史被改写）|
| **Type 2** ⭐ | **插入新行**，旧行标记失效 | ✅ 完整保留 | US（正确）|
| Type 3 | 加一列 `prev_country` | ⚠ 只留一步 | 看你查哪列 |

**Type 2 是分析工程的标准答案**。每行维度记录带"生效区间"：
Type 2 is the analytics-engineering standard. Each dimension row carries a validity interval:


In [ ]:
# ---- 亲手实现 SCD Type 2 / Implement SCD Type 2 by hand ----
conn.sql("""
CREATE TABLE dim_customer_scd (
    customer_key   INT,            -- 代理键：每个版本一个新 key！/ new key per VERSION
    customer_nk    INT,            -- 业务键：版本之间相同 / same across versions
    name           VARCHAR,
    country        VARCHAR,
    valid_from     DATE,
    valid_to       DATE,           -- '9999-12-31' = 当前版本 / current
    is_current     BOOLEAN
);

-- 初始装载：Alice 在 US / Initial load
INSERT INTO dim_customer_scd VALUES
    (1, 101, 'Alice Chen', 'US', DATE '2025-01-01', DATE '9999-12-31', TRUE);
""")
print("--- 初始状态 / Initial ---")
print(conn.sql("SELECT * FROM dim_customer_scd").df())


In [ ]:
# 2026-03-10: Alice 搬家 US → CA。SCD2 两步操作：
# Alice moves on 2026-03-10. SCD2 = two steps:

# Step 1: 关闭旧版本 / Close the old version
conn.sql("""
    UPDATE dim_customer_scd
    SET valid_to = DATE '2026-03-09', is_current = FALSE
    WHERE customer_nk = 101 AND is_current;
""")

# Step 2: 插入新版本（新代理键！）/ Insert the new version with a NEW surrogate key
conn.sql("""
    INSERT INTO dim_customer_scd VALUES
    (2, 101, 'Alice Chen', 'CA', DATE '2026-03-10', DATE '9999-12-31', TRUE);
""")

print("--- 搬家后：同一个人两行 / After the move: two rows, one person ---")
print(conn.sql("SELECT * FROM dim_customer_scd ORDER BY valid_from").df())


In [ ]:
# ---- SCD2 的回报：历史正确归属 / The payoff: correct historical attribution ----
# Alice 的两笔订单：1 月一笔、4 月一笔
conn.sql("""
CREATE TABLE alice_orders (order_id INT, customer_nk INT, order_date DATE, amount DECIMAL(8,2));
INSERT INTO alice_orders VALUES
    (1, 101, DATE '2026-01-15', 50.00),     -- 搬家前 / before the move
    (2, 101, DATE '2026-04-20', 80.00);     -- 搬家后 / after the move
""")

# as-of JOIN：按订单日期落进哪个"生效区间"来挂维度 ⭐
# The as-of JOIN: attach the dimension version whose interval contains the order date
print(conn.sql("""
    SELECT
        o.order_id,
        o.order_date,
        o.amount,
        d.country AS country_at_order_time     -- ← 历史时点的真实国家
    FROM alice_orders o
    JOIN dim_customer_scd d
      ON o.customer_nk = d.customer_nk
     AND o.order_date BETWEEN d.valid_from AND d.valid_to
    ORDER BY o.order_date;
""").df())


**完美**：1 月订单归 **US**，4 月订单归 **CA**——历史没有被改写。
January order attributes to US, April to CA — history intact.

这个 `BETWEEN valid_from AND valid_to` 的 JOIN 叫 **as-of join**，是数仓查询的标志性模式。
The `BETWEEN valid_from AND valid_to` join is the **as-of join** — a signature warehouse pattern.

> 💡 **面试满分点**：能说出"SCD2 每个版本要发**新代理键**，事实表存的是当时的版本 key，所以事实行天然钉死在历史版本上，连 as-of join 都省了"——这是 Kimball 的完整设计。
> Full-credit answer: each SCD2 version gets a NEW surrogate key; fact rows store the version key current at load time, pinning facts to history without even needing the as-of join.


<a id="8"></a>
## 8. 雪花模型 & One Big Table / Snowflake Schema & OBT

### 雪花模型 / Snowflake schema

把维度**再规范化**一层：`dim_track → dim_album → dim_artist` 拆开。形状像雪花。
Re-normalize the dimensions one more level — shaped like a snowflake.

| | 星型 / Star | 雪花 / Snowflake |
|---|---|---|
| 维度 | 拍平（冗余）| 规范化（无冗余）|
| JOIN 数 | 少 ⭐ | 多 |
| 存储 | 略多（无所谓）| 略省 |
| 业务可读性 | 高 ⭐ | 低 |

**工业共识：默认星型**。存储便宜到冗余几个 GB 无感，但每多一层 JOIN，查询和心智成本都涨。
Industry consensus: default to star. Storage is too cheap to care; every extra JOIN costs query and cognitive overhead.

### One Big Table (OBT)：更极端的反范式

干脆**把事实和所有维度全 JOIN 成一张宽表**物化下来——列存引擎里"宽"几乎免费（不读的列不扫）。
JOIN everything into one materialized wide table — "wide" is nearly free in columnar engines (unread columns aren't scanned).

```
谱系 / The spectrum:
  雪花 ──────── 星型 ──────── OBT
  最规范化      平衡 ⭐        最反范式
  (少见)       (默认)         (BI 自助层流行, dbt 时代常见)
```

现代常见做法：**核心层星型（可治理）→ 给 BI 的 mart 层 OBT（好用）**——下一课 dbt 就是干这个的工具。
Modern pattern: star at the core (governable) → OBT marts for BI (usable). dbt — next lesson — is the tool that builds these layers.


<a id="9"></a>
## 9. ETL vs ELT

数据从业务库到数仓的两种路线：
Two routes from OLTP to warehouse:

```
ETL (老派 / classic):
  Extract → Transform (专用服务器, Informatica/SSIS) → Load 干净数据进仓
  逻辑藏在 ETL 工具里，黑盒，难版本控制

ELT (现代 / modern) ⭐:
  Extract → Load 原始数据直接进仓 (Fivetran/Airbyte)
          → Transform 用 SQL 在仓内完成 (dbt)
  逻辑就是 SQL 文件 → git 管理、code review、可测试
```

### 为什么倒向 ELT / Why the flip happened

1. **云数仓算力便宜且弹性**——转换在仓内跑比专用 ETL 服务器划算
2. **原始数据落仓**——转换逻辑错了随时重放（ETL 时代源数据没存，错了就没了）
3. **SQL 民主化**——分析师自己写转换（dbt），不用排队等数据工程师

> 💡 现代数据栈一句话：**Fivetran/Airbyte (EL) + Snowflake/BigQuery (仓) + dbt (T) + BI**。


<a id="10"></a>
## 10. 云数仓三巨头 / The Big-Three Cloud Warehouses

| | Snowflake | BigQuery | Redshift |
|---|---|---|---|
| 厂商 | 独立（跨云）| Google | AWS |
| 架构卖点 | **存算分离**：多仓共享一份数据，互不抢资源 | **Serverless**：零运维，按扫描量计费 | 与 AWS 生态深度绑定 |
| 计费 | 按仓运行时间（秒级）| 按扫描字节（$/TB）| 按节点（或 serverless）|
| 一句话 | "多团队并发分析的企业默认" | "不想管任何基础设施" | "已经全家在 AWS" |

> ⚠ **BigQuery 按扫描计费的坑**：`SELECT *` 扫全表列 = 烧钱。**选列、用分区表过滤**直接省 10-100× 账单——又一次回到"列存只读所需列"的原理。
> BigQuery bills by bytes scanned: `SELECT *` burns money. Selecting columns + partition filters cuts bills 10-100x — the columnar principle, now with a price tag.

**DuckDB 的定位**：同样的列存 + 向量化技术，跑在你笔记本上，$0。**中小数据（< 100 GB）的"个人数仓"**——本课程全程用它不是偶然。
DuckDB: same columnar tech on your laptop for $0 — the "personal warehouse" for <100 GB. Using it throughout this course is no accident.


<a id="11"></a>
## 11. 小结 / Summary

### 概念地图 / Concept map

```
数仓 / Warehouse
  │
  ├── OLTP vs OLAP → 行存 vs 列存（+ 压缩 10-30× + zone maps）
  │
  ├── 星型模型 ⭐
  │     ├── fact: 可加数字 + 外键，行数巨大
  │     ├── dim: 描述属性，故意反范式
  │     ├── 粒度: 第一决定，最细可行粒度，一表一粒度
  │     └── 代理键: *_key (内部) vs *_nk (业务)
  │
  ├── SCD ⭐
  │     ├── Type 1 覆盖（丢历史）
  │     ├── Type 2 新行 + valid_from/to + is_current（标准答案）
  │     └── as-of join: BETWEEN valid_from AND valid_to
  │
  ├── 形状谱系: 雪花 ← 星型(默认) → OBT(BI 层)
  │
  └── ETL → ELT: 原始数据落仓, SQL 转换 (dbt), git 管理
```

### 💡 面试速查 / Interview must-knows

1. **星型 vs 雪花**：维度拍平 vs 再规范化；默认星型（JOIN 少、可读）
2. **粒度**：一行代表什么；先声明；取最细；一表一粒度
3. **代理键三理由**：业务键会变 / 多源合并 / SCD2 需要多版本
4. **SCD2 操作**：关旧行（valid_to + is_current=F）→ 插新行（新代理键）
5. **ELT 取代 ETL**：云仓算力 + 原始数据可重放 + SQL 民主化
6. **BigQuery 省钱**：永远不 `SELECT *`，用分区过滤

### 下一节预告 / Next up

**Part 1.11 · dbt 入门** —— Part 1 收官。ELT 里的 "T"：用 SQL 文件 + git 管理整个数仓的转换层，model / ref / test / lineage。
**Part 1.11 · dbt Intro** — Part 1 finale. The "T" in ELT: models, refs, tests, lineage.
